# Notebook 19 — N17v2 frozen V3 biometric evaluation and FER provenance gate

**PATCHED UNEXECUTED — fixed N12 exact SHA-pinned InsightFace wheel URL.** No generation; all outcomes pending this run.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, pandas as pd

ROOT = Path('/content/drive/MyDrive/thesis_all/outputs/runs')

N07 = ROOT / '20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664'
N09 = ROOT / '20261013_nb09_detector-rootcause_4fae269d_3a2979d8_20261013T201108057643Z_02a058fe' / 'sessions' / '20261013T201108057643Z_02a058fe'
N12 = ROOT / 'nb12_v3_biometric_11faafd9beb1_20261013T211442695856Z_bcf37701'
N17 = ROOT / 'nb17v2_neutral_expression_12sources_2seeds_sd3pinned'
N18 = ROOT / 'nb18_paired_neutral_pilot_analysis_v1'

OUT = ROOT / 'nb19_n17v2_frozen_v3_biometrics'
OUT.mkdir(exist_ok=True)

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

rec = pd.read_csv(N17 / 'private_execution_receipts.csv')
paired = pd.read_csv(N18 / 'private_paired_reference_and_n17_manifest.csv')
niqe = pd.read_csv(N18 / 'private_n17_niqe.csv')

assert len(rec) == len(paired) == len(niqe) == 24
assert rec.source_id.nunique() == 12

for row in rec.itertuples(index=False):
    assert sha(row.output_path) == row.output_sha256

EXPECTED = {
    'detector': '5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91',
    'arcface': '4c06341c33c2ca1f86781dab0e829f88ad5b64be9fba56e56bc9ebdefc619e43',
    'auraface': 'a7933ea5330113b01c9b60351d8f4c33003f145d8470ac5f0e52ee2effe25c60'
}

archive = json.loads((N09 / 'model_file_hashes.json').read_text())

MODELS = {}
for name, digest in EXPECTED.items():
    matches = [
        Path(path)
        for path, saved_hash in archive.items()
        if saved_hash == digest and Path(path).is_file()
    ]
    assert matches, f'Missing frozen model: {name}'
    MODELS[name] = matches[0]
    assert sha(MODELS[name]) == digest

print('SETUP OK — 24 images and frozen model hashes verified.')


In [ ]:
!pip -q install 'numpy==1.26.4' 'pandas==2.2.2' 'Pillow==11.3.0' 'onnxruntime==1.19.2' 'https://files.pythonhosted.org/packages/42/ce/33989427a7fae952e1606d0edee4191f965630f3c74f2166fa776fafd271/insightface-1.0.1-py3-none-any.whl#sha256=5f373f6fedbdda5cbc59a34ca386a75a2995cdaf6899402590ae9eb4308fc2e8' 'opencv-python==4.11.0.86' 'scikit-image==0.25.2'
import importlib.metadata as _metadata
assert _metadata.version('insightface') == '1.0.1', 'Frozen N12 InsightFace version differs'
from google.colab import drive
drive.mount('/content/drive')
import hashlib,json,sys,platform,subprocess,os
from pathlib import Path
import numpy as np,pandas as pd,cv2
from PIL import Image
from insightface.model_zoo import get_model
from insightface.utils import face_align
assert platform.python_version().startswith('3.12.'), 'Original N12 runtime was Python 3.12'
ROOT=Path('/content/drive/MyDrive/thesis_all/outputs/runs')
N07=ROOT/'20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664'
N09=ROOT/'20261013_nb09_detector-rootcause_4fae269d_3a2979d8_20261013T201108057643Z_02a058fe'/'sessions'/'20261013T201108057643Z_02a058fe'
N12=ROOT/'nb12_v3_biometric_11faafd9beb1_20261013T211442695856Z_bcf37701'
N17=ROOT/'nb17v2_neutral_expression_12sources_2seeds_sd3pinned'
N18=ROOT/'nb18_paired_neutral_pilot_analysis_v1'
OUT=ROOT/'nb19_n17v2_frozen_v3_biometrics';OUT.mkdir(exist_ok=True)
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
 return h.hexdigest()
rec=pd.read_csv(N17/'private_execution_receipts.csv')
paired=pd.read_csv(N18/'private_paired_reference_and_n17_manifest.csv')
niqe=pd.read_csv(N18/'private_n17_niqe.csv')
assert len(rec)==len(paired)==len(niqe)==24 and rec.source_id.nunique()==12
assert not rec.duplicated(['source_id','seed']).any()
assert set(rec.status)=={'VERIFIED'}
for r in rec.itertuples(index=False):assert Path(r.output_path).is_file() and sha(r.output_path)==r.output_sha256
EXPECTED={'detector':'5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91','arcface':'4c06341c33c2ca1f86781dab0e829f88ad5b64be9fba56e56bc9ebdefc619e43','auraface':'a7933ea5330113b01c9b60351d8f4c33003f145d8470ac5f0e52ee2effe25c60'}
archive=json.loads((N09/'model_file_hashes.json').read_text())
MODELS={}
for k,digest in EXPECTED.items():
 candidates=[Path(p) for p,h in archive.items() if h==digest and Path(p).is_file()]
 assert candidates,f'N12 archived weight missing: {k}; do not substitute another model'
 MODELS[k]=candidates[0]
 assert sha(MODELS[k])==digest
print('INPUT + EXACT WEIGHT HASHES VERIFIED:',len(rec),'pairs')


In [ ]:
# Reuse frozen N12 V3 detection and 112x112 alignment exactly.
det=get_model(str(MODELS['detector']),providers=['CPUExecutionProvider'])
assert type(det).__name__=='SCRFD'
det.prepare(ctx_id=-1,input_size=(640,640),det_thresh=0.1)
recognizers={}
for name in ('arcface','auraface'):
 mod=get_model(str(MODELS[name]),providers=['CPUExecutionProvider']);mod.prepare(ctx_id=-1)
 recognizers[name]=mod

def extract(path):
 rgb=np.asarray(Image.open(path).convert('RGB'))
 assert rgb.shape==(512,512,3)
 bgr=cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)
 canvas=np.zeros((640,640,3),dtype=np.uint8)
 canvas[160:480,160:480]=cv2.resize(bgr,(320,320),interpolation=cv2.INTER_AREA)
 boxes,lm=det.detect(canvas,max_num=0,metric='default')
 result={'face_detected':bool(len(boxes)),'alignment_plausible':False,'embedding_valid_arcface':False,'embedding_valid_auraface':False}
 embeddings={}
 if not len(boxes):return result,embeddings
 ix=int(np.argmax(boxes[:,4]));b=np.asarray(boxes[ix],dtype=np.float64)
 pts=(np.asarray(lm[ix],dtype=np.float64)-160)/0.625
 xy=(b[:4]-160)/0.625;x1,y1,x2,y2=xy
 area=max(0,x2-x1)*max(0,y2-y1)
 overlap=max(0,min(512,x2)-max(0,x1))*max(0,min(512,y2)-max(0,y1))
 fraction=overlap/area if area>0 else 0.
 plausible=bool(pts.shape==(5,2) and np.isfinite(pts).all() and np.linalg.norm(pts[0]-pts[1])>=1 and fraction>=0.7)
 aligned=face_align.norm_crop(bgr,landmark=pts.astype(np.float32),image_size=112)
 result['alignment_plausible']=bool(aligned.shape==(112,112,3) and aligned.dtype==np.uint8 and plausible)
 for name,model in recognizers.items():
  vec=np.asarray(model.get_feat(aligned),dtype=np.float32).reshape(-1)
  magnitude=float(np.linalg.norm(vec))
  valid=(vec.size==512 and np.isfinite(vec).all() and np.isfinite(magnitude) and magnitude>1e-10 and result['alignment_plausible'])
  result['embedding_valid_'+name]=bool(valid)
  if valid:embeddings[name]=vec/magnitude
 return result,embeddings


In [ ]:
# Each completed job committed privately; no model output or biometric row is published.
CACHE=OUT/'private_v3_biometric_rows.csv'
records=[]
for ix,row in enumerate(rec.sort_values(['source_id','seed']).itertuples(index=False),1):
 if CACHE.exists():
  prior=pd.read_csv(CACHE)
  same=prior[(prior.source_id==row.source_id)&(prior.seed==row.seed)&(prior.output_sha256==row.output_sha256)]
  if len(same)==1:
   records.append(same.iloc[0].to_dict());print(f'{ix}/24 cached',flush=True);continue
 source=N07/'sources'/f'{row.source_id}.png'
 assert sha(source)==row.source_sha256
 st,se=extract(source)
 gt,ge=extract(row.output_path)
 record={'source_id':row.source_id,'seed':int(row.seed),'output_sha256':row.output_sha256,'source_sha256':row.source_sha256,
 'source_detected':st['face_detected'],'output_detected':gt['face_detected'],
 'source_alignment_plausible':st['alignment_plausible'],'output_alignment_plausible':gt['alignment_plausible']}
 for name in ('arcface','auraface'):
  valid=name in se and name in ge
  record['pair_valid_'+name]=bool(valid)
  record['n17_'+name+'_cosine']=float(np.dot(se[name],ge[name])) if valid else np.nan
 records.append(record)
 pd.DataFrame(records).to_csv(CACHE,index=False)
 print(f'{ix}/24 evaluated: {row.source_id} seed {row.seed}',flush=True)
biom=pd.read_csv(CACHE)
assert len(biom)==24 and not biom.duplicated(['source_id','seed']).any()
assert set(biom.output_sha256)==set(rec.output_sha256)
print('Valid ArcFace:',int(biom.pair_valid_arcface.sum()),'of 24; AuraFace:',int(biom.pair_valid_auraface.sum()),'of 24')


In [ ]:
# Merge source/seed 1:1, NIQE, and N12 reference; never fill invalid cosine as zero.
keys=['source_id','seed']
full=paired.merge(niqe[keys+['output_sha256','n17_niqe']],on=keys+['output_sha256'],validate='one_to_one')
full=full.merge(biom,on=keys+['output_sha256','source_sha256'],validate='one_to_one')
assert len(full)==24
full['delta_niqe']=full.n17_niqe-full.n07_niqe
for k in ['arcface','auraface']:
 full['delta_'+k]=full['n17_'+k+'_cosine']-full['n07_'+k+'_cosine']
rng=np.random.default_rng(19019)
def cluster_ci(data,col,B=10000):
 groups=[g[col].to_numpy(dtype=float) for _,g in data.groupby('source_id',sort=True)]
 draws=[]
 for _ in range(B):
  sampled=rng.integers(0,len(groups),len(groups))
  z=np.concatenate([groups[i] for i in sampled]);draws.append(float(np.nanmean(z)) if np.isfinite(z).any() else np.nan)
 return np.nanquantile(draws,[.025,.975]).tolist()
summary=[]
for col in ['delta_niqe','delta_arcface','delta_auraface']:
 vals=full[col].dropna()
 ci=cluster_ci(full,col) if len(vals) else [np.nan,np.nan]
 summary.append({'metric':col,'n_valid_pairs':len(vals),'n_sources':full.source_id.nunique(),'mean_delta':float(vals.mean()) if len(vals) else np.nan,'ci95_low':ci[0],'ci95_high':ci[1]})
full.to_csv(OUT/'private_paired_biometrics_and_niqe.csv',index=False)
pd.DataFrame(summary).to_csv(OUT/'private_paired_biometrics_summary.csv',index=False)
print(pd.DataFrame(summary).round(4).to_string(index=False))
print('NOTE: FER NOT YET RE-EVALUATED; exact N07 classifier must be replayed before N18 final status changes.')
status={'n17_biometrics':'EXECUTED' if biom.pair_valid_arcface.all() and biom.pair_valid_auraface.all() else 'PARTIAL_REQUIRES_REVIEW',
'n17_emotion':'PENDING_EXACT_N07_CLASSIFIER','source_count':12,'pair_count':24,'manual_alignment_review':'PENDING',
'reidentification_calibrated':False,'source':'N12 frozen V3 detector+embedding weights and geometry'}
(OUT/'status.json').write_text(json.dumps(status,indent=2))
